### Amazon Sentiment Data

In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import lxmls.readers.sentiment_reader as srs
from lxmls.deep_learning.utils import AmazonData
corpus = srs.SentimentCorpus("books")
data = AmazonData(corpus=corpus)

### Implement Pytorch Forward pass

As the final exercise today implement the log `forward()` method in 

    lxmls/deep_learning/pytorch_models/mlp.py

Use the previous exercise as reference. After you have completed this you can run both systems for comparison.

In [2]:
# Model
geometry = [corpus.nr_features, 20, 2]
activation_functions = ['sigmoid', 'softmax']

# Optimization
learning_rate = 0.05
num_epochs = 10
batch_size = 30

In [22]:
import numpy as np
import numpy as np
import torch
from lxmls.deep_learning.mlp import MLP


def cast_float(variable_np):
    variable = torch.from_numpy(variable_np).float()
    variable.requires_grad = True
    return variable


class PytorchMLP(MLP):
    """
    Basic MLP with forward-pass and gradient computation in Pytorch
    """

    def __init__(self, **config):

        # This will initialize
        # self.num_layers
        # self.config
        # self.parameters
        MLP.__init__(self, **config)

        # Need to cast all weights
        for n in range(self.num_layers):
            # Get weigths and bias of the layer (even and odd positions)
            weight, bias = self.parameters[n]
            self.parameters[n] = [cast_float(weight), cast_float(bias)]

        # Initialize some functions that we will need
        self.log_softmax = torch.nn.LogSoftmax(dim=1)
        self.loss_function = torch.nn.NLLLoss()

    # TODO: Move these outside fo the class as in the numpy case
    def _log_forward(self, input):
        """
        Forward pass
        """

        # Ensure the type matches torch type
        input = cast_float(input)

        # Input
        tilde_z = input

        # ----------
        # Solution to Exercise 4

        num_hidden_layers = len(self.parameters) - 1
        for n in range(num_hidden_layers):
            # Linear transformation
            weight, bias = self.parameters[n]
            z = tilde_z@ weight.T + bias

            # Non-linear transformation (sigmoid)
            tilde_z = 1.0 / (1 + torch.exp(-z))


        # Output linear transformation
        weight, bias = self.parameters[num_hidden_layers]
        z = tilde_z@ weight.T + bias

        return self.log_softmax(z)
        # End of solution to Exercise 4
        # ----------


    def gradients(self, input, output):
        """
        Computes the gradients of the network with respect to cross entropy
        error cost
        """
        true_class = torch.from_numpy(output).long()

        # Compute negative log-likelihood loss
        _log_forward = self._log_forward(input)
        loss = self.loss_function(_log_forward, true_class)
        # Use autograd to compute the backward pass.
        loss.backward()

        nabla_parameters = []
        for n in range(self.num_layers):
            weight, bias = self.parameters[n]
            nabla_parameters.append([weight.grad.data, bias.grad.data])
        return nabla_parameters

    def predict(self, input=None):
        """
        Predict model outputs given input
        """
        log_forward = self._log_forward(input).data.numpy()
        return np.argmax(log_forward, axis=1)

    def update(self, input=None, output=None):
        """
        Update model parameters given batch of data
        """
        gradients = self.gradients(input, output)
        learning_rate = self.config['learning_rate']
        # Update each parameter with SGD rule
        for m in range(self.num_layers):
            # Update weight
            self.parameters[m][0].data -= learning_rate * gradients[m][0]
            # Update bias
            self.parameters[m][1].data -= learning_rate * gradients[m][1]

        # Zero gradients
        for n in range(self.num_layers):
            weight, bias = self.parameters[n]
            weight.grad.data.zero_()
            bias.grad.data.zero_()

model = PytorchMLP(
    geometry=geometry,
    activation_functions=activation_functions,
    learning_rate=learning_rate
)

In [23]:
# Get batch iterators for train and test
train_batches = data.batches('train', batch_size=batch_size)
test_set = data.batches('test', batch_size=None)[0]

# Epoch loop
for epoch in range(num_epochs):

    # Batch loop
    for batch in train_batches:
        model.update(input=batch['input'], output=batch['output'])

    # Prediction for this epoch
    hat_y = model.predict(input=test_set['input'])

    # Evaluation
    accuracy = 100*np.mean(hat_y == test_set['output'])

    # Inform user
    print("Epoch %d: accuracy %2.2f %%" % (epoch+1, accuracy))

Epoch 1: accuracy 60.25 %
Epoch 2: accuracy 67.50 %
Epoch 3: accuracy 74.50 %
Epoch 4: accuracy 76.25 %
Epoch 5: accuracy 78.00 %
Epoch 6: accuracy 78.75 %
Epoch 7: accuracy 79.00 %
Epoch 8: accuracy 79.75 %
Epoch 9: accuracy 80.00 %
Epoch 10: accuracy 80.25 %
